# Análisis de episodios hospitalarios vs pacientes únicos

Este notebook permite responder la observación metodológica sobre si el análisis de la tesis fue realizado a nivel de **episodio hospitalario** o de **paciente único**.

Archivo esperado:

```python
'../1_data_processed/v1_base_filas_validas.csv'
```

Columna identificadora:

```python
'ID_BENEFICIARIO'
```

El objetivo es cuantificar:

- total de registros o episodios;
- total de beneficiarios únicos;
- cantidad de episodios por beneficiario;
- beneficiarios con más de un episodio;
- posible implicancia metodológica para independencia entre observaciones.


In [ ]:
# =========================================================
# CONFIGURACIÓN INICIAL
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np

RUTA_DATASET = Path("../1_data_processed/v1_base_filas_validas.csv")
COL_ID = "ID_BENEFICIARIO"

print("Ruta dataset:", RUTA_DATASET)
print("Existe archivo:", RUTA_DATASET.exists())


In [ ]:
# =========================================================
# CARGA DEL DATASET
# =========================================================
# Se carga solo la columna ID_BENEFICIARIO para reducir uso de memoria.
# Esto es suficiente para analizar si los registros corresponden a episodios
# repetidos por paciente/beneficiario.

df_id = pd.read_csv(
    RUTA_DATASET,
    usecols=[COL_ID],
    dtype={COL_ID: "string"},
    low_memory=False
)

df_id.head()


In [ ]:
# =========================================================
# VALIDACIÓN BÁSICA
# =========================================================

total_registros = len(df_id)
nulos_id = df_id[COL_ID].isna().sum()
ids_unicos = df_id[COL_ID].nunique(dropna=True)

print(f"Total de registros / episodios: {total_registros:,}")
print(f"ID_BENEFICIARIO nulos: {nulos_id:,}")
print(f"Beneficiarios únicos: {ids_unicos:,}")

if ids_unicos > 0:
    print(f"Promedio de episodios por beneficiario: {total_registros / ids_unicos:.4f}")


In [ ]:
# =========================================================
# DISTRIBUCIÓN DE EPISODIOS POR BENEFICIARIO
# =========================================================

episodios_por_paciente = (
    df_id
    .dropna(subset=[COL_ID])
    .groupby(COL_ID, observed=True)
    .size()
    .rename("n_episodios")
    .reset_index()
)

episodios_por_paciente.head()


In [ ]:
# =========================================================
# RESUMEN DE REPETICIÓN DE BENEFICIARIOS
# =========================================================

beneficiarios_con_1 = (episodios_por_paciente["n_episodios"] == 1).sum()
beneficiarios_con_mas_de_1 = (episodios_por_paciente["n_episodios"] > 1).sum()

porc_beneficiarios_repetidos = beneficiarios_con_mas_de_1 / ids_unicos * 100 if ids_unicos else np.nan

episodios_de_beneficiarios_repetidos = episodios_por_paciente.loc[
    episodios_por_paciente["n_episodios"] > 1,
    "n_episodios"
].sum()

porc_episodios_de_repetidos = episodios_de_beneficiarios_repetidos / total_registros * 100 if total_registros else np.nan

resumen = pd.DataFrame({
    "Indicador": [
        "Total registros / episodios",
        "Beneficiarios únicos",
        "Promedio episodios por beneficiario",
        "Beneficiarios con 1 episodio",
        "Beneficiarios con más de 1 episodio",
        "% beneficiarios con más de 1 episodio",
        "Episodios pertenecientes a beneficiarios repetidos",
        "% episodios pertenecientes a beneficiarios repetidos",
        "ID_BENEFICIARIO nulos"
    ],
    "Valor": [
        total_registros,
        ids_unicos,
        total_registros / ids_unicos if ids_unicos else np.nan,
        beneficiarios_con_1,
        beneficiarios_con_mas_de_1,
        porc_beneficiarios_repetidos,
        episodios_de_beneficiarios_repetidos,
        porc_episodios_de_repetidos,
        nulos_id
    ]
})

resumen


In [ ]:
# =========================================================
# ESTADÍSTICOS DESCRIPTIVOS
# =========================================================

estadisticos = episodios_por_paciente["n_episodios"].describe(
    percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]
)

estadisticos


In [ ]:
# =========================================================
# FRECUENCIA DE CANTIDAD DE EPISODIOS POR BENEFICIARIO
# =========================================================

frecuencia_episodios = (
    episodios_por_paciente["n_episodios"]
    .value_counts()
    .sort_index()
    .rename_axis("n_episodios")
    .reset_index(name="cantidad_beneficiarios")
)

frecuencia_episodios["porcentaje_beneficiarios"] = (
    frecuencia_episodios["cantidad_beneficiarios"] / ids_unicos * 100
)

frecuencia_episodios.head(20)


In [ ]:
# =========================================================
# TOP BENEFICIARIOS CON MÁS EPISODIOS
# =========================================================
# Se recomienda no incluir esta tabla completa en la tesis para evitar exponer IDs.
# Puede usarse solo como diagnóstico interno.

top_repetidos = episodios_por_paciente.sort_values("n_episodios", ascending=False).head(20)
top_repetidos


In [ ]:
# =========================================================
# TABLA RESUMEN PARA TESIS
# =========================================================
# Esta tabla está pensada para copiarse al documento,
# sin exponer identificadores individuales.

tabla_tesis = pd.DataFrame({
    "Métrica": [
        "Total de episodios hospitalarios",
        "Total de beneficiarios únicos",
        "Promedio de episodios por beneficiario",
        "Beneficiarios con más de un episodio",
        "Porcentaje de beneficiarios con más de un episodio",
        "Porcentaje de episodios asociados a beneficiarios repetidos"
    ],
    "Valor": [
        f"{total_registros:,}",
        f"{ids_unicos:,}",
        f"{total_registros / ids_unicos:.4f}" if ids_unicos else "No calculable",
        f"{beneficiarios_con_mas_de_1:,}",
        f"{porc_beneficiarios_repetidos:.2f}%" if not np.isnan(porc_beneficiarios_repetidos) else "No calculable",
        f"{porc_episodios_de_repetidos:.2f}%" if not np.isnan(porc_episodios_de_repetidos) else "No calculable"
    ]
})

tabla_tesis


In [ ]:
# =========================================================
# EXPORTAR RESULTADOS
# =========================================================

CARPETA_SALIDA = Path("../4_results/analisis_episodios_pacientes")
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

resumen.to_csv(CARPETA_SALIDA / "resumen_episodios_vs_pacientes.csv", index=False, encoding="utf-8-sig")
estadisticos.to_frame("valor").to_csv(CARPETA_SALIDA / "estadisticos_episodios_por_paciente.csv", encoding="utf-8-sig")
frecuencia_episodios.to_csv(CARPETA_SALIDA / "frecuencia_episodios_por_paciente.csv", index=False, encoding="utf-8-sig")
tabla_tesis.to_csv(CARPETA_SALIDA / "tabla_tesis_episodios_vs_pacientes.csv", index=False, encoding="utf-8-sig")

print("Archivos exportados en:", CARPETA_SALIDA)


## Interpretación sugerida para el documento

Después de ejecutar el notebook, reemplaza los valores `X` por los resultados obtenidos.

### Texto sugerido para sección 4.2

> El análisis se realizó a nivel de episodio hospitalario, considerando cada registro como una observación del conjunto de datos. A partir de la variable `ID_BENEFICIARIO`, se identificaron X beneficiarios únicos frente a X episodios hospitalarios, con un promedio de X episodios por beneficiario. Esto indica que un mismo beneficiario puede estar representado por más de un episodio dentro de la base de datos, aspecto relevante para la interpretación metodológica del estudio.

### Texto sugerido para sección 7.2 Limitaciones

> Una limitación metodológica del estudio corresponde a que el análisis fue realizado a nivel de episodio hospitalario y no estrictamente a nivel de paciente único. En consecuencia, un mismo beneficiario puede aparecer en más de un registro, lo que reduce parcialmente la independencia estadística entre observaciones. Asimismo, si la partición de datos fue realizada a nivel de episodio, existe la posibilidad de que episodios pertenecientes a un mismo beneficiario hayan quedado distribuidos entre subconjuntos de entrenamiento y evaluación. Esta condición debe considerarse al interpretar los resultados, aunque el bajo desempeño predictivo observado sugiere que dicha posible dependencia no produjo una señal favorable para los modelos.
